In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from feature_eng import preprocess,correlation_filter
from sklearn.metrics import root_mean_squared_error
df = pd.read_csv("~/ML/Data/house_prices/train.csv")
x_train,x_test = train_test_split(df,test_size=0.2,random_state=42)

x_train,x_test,y_train,y_test = preprocess(x_train,x_test)

In [2]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV,KFold,cross_val_score

kf = KFold(n_splits=5,random_state=42,shuffle=True)

param_grid = {
    'max_depth': [3, 5, 7, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10, 20, 50],
    'min_samples_leaf': [1, 5, 10, 20, 50],
    'max_features': [0.5, 0.75, 1.0, 'sqrt', 'log2'],
}

search = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid,
    cv=kf,
    scoring='neg_mean_squared_error',
    return_train_score=True,
    verbose=2,
)


In [5]:
result_basic = search.fit(x_train,y_train)

Fitting 5 folds for each of 875 candidates, totalling 4375 fits
[CV] END max_depth=3, max_features=0.5, min_samples_leaf=1, min_samples_split=2; total time=   0.0s
[CV] END max_depth=3, max_features=0.5, min_samples_leaf=1, min_samples_split=2; total time=   0.0s
[CV] END max_depth=3, max_features=0.5, min_samples_leaf=1, min_samples_split=2; total time=   0.0s
[CV] END max_depth=3, max_features=0.5, min_samples_leaf=1, min_samples_split=2; total time=   0.0s
[CV] END max_depth=3, max_features=0.5, min_samples_leaf=1, min_samples_split=2; total time=   0.0s
[CV] END max_depth=3, max_features=0.5, min_samples_leaf=1, min_samples_split=5; total time=   0.0s
[CV] END max_depth=3, max_features=0.5, min_samples_leaf=1, min_samples_split=5; total time=   0.0s
[CV] END max_depth=3, max_features=0.5, min_samples_leaf=1, min_samples_split=5; total time=   0.0s
[CV] END max_depth=3, max_features=0.5, min_samples_leaf=1, min_samples_split=5; total time=   0.0s
[CV] END max_depth=3, max_features=0

In [9]:
best_idx = result_basic.best_index_
train_rmse = np.sqrt(-result_basic.cv_results_['mean_train_score'][best_idx])
val_rmse = np.sqrt(-result_basic.best_score_)
val_std = result_basic.cv_results_['std_test_score'][best_idx]
overfit_gap = np.abs(train_rmse - val_rmse)
test_pred_basic = result_basic.best_estimator_.predict(x_test)
test_rmse_basic = root_mean_squared_error(y_test,test_pred_basic)
print(train_rmse,val_rmse,val_std,overfit_gap,test_rmse)

0.1445450491432947 0.19096145046550006 0.0045491077916222375 0.04641640132220537 0.19813052030462516


In [6]:
accuracy = np.sqrt(-result_basic.score(x_test,y_test))
print("Test Set Accuracy",accuracy)

Test Set Accuracy 0.19813052030462516


# Running model with filtered data


In [11]:
x_filtered = correlation_filter(x_train,y_train,0.7)

In [12]:
result_filtered = search.fit(x_filtered,y_train)


Fitting 5 folds for each of 875 candidates, totalling 4375 fits
[CV] END max_depth=3, max_features=0.5, min_samples_leaf=1, min_samples_split=2; total time=   0.0s
[CV] END max_depth=3, max_features=0.5, min_samples_leaf=1, min_samples_split=2; total time=   0.0s
[CV] END max_depth=3, max_features=0.5, min_samples_leaf=1, min_samples_split=2; total time=   0.0s
[CV] END max_depth=3, max_features=0.5, min_samples_leaf=1, min_samples_split=2; total time=   0.0s
[CV] END max_depth=3, max_features=0.5, min_samples_leaf=1, min_samples_split=2; total time=   0.0s
[CV] END max_depth=3, max_features=0.5, min_samples_leaf=1, min_samples_split=5; total time=   0.0s
[CV] END max_depth=3, max_features=0.5, min_samples_leaf=1, min_samples_split=5; total time=   0.0s
[CV] END max_depth=3, max_features=0.5, min_samples_leaf=1, min_samples_split=5; total time=   0.0s
[CV] END max_depth=3, max_features=0.5, min_samples_leaf=1, min_samples_split=5; total time=   0.0s
[CV] END max_depth=3, max_features=0

In [16]:
x_test_filtered = x_test[x_filtered.columns]
best_idx = result_filtered.best_index_
train_rmse = np.sqrt(-result_filtered.cv_results_['mean_train_score'][best_idx])
val_rmse = np.sqrt(-result_filtered.best_score_)
val_std = result_filtered.cv_results_['std_test_score'][best_idx]
overfit_gap = np.abs(train_rmse - val_rmse)
test_pred_filtered = result_filtered.best_estimator_.predict(x_test_filtered)
test_rmse_filtered = root_mean_squared_error(y_test,test_pred_filtered)
print("train_rmse: ",train_rmse,"\nval_rmse: ",val_rmse,"\nval_std: " ,val_std,"\noverfit gap",overfit_gap)

train_rmse:  0.13417443056911296 
val_rmse:  0.1892398968313218 
val_std:  0.003912149059299312 
overfit gap 0.055065466262208845


## MLflow logging

In [ ]:
!pip install

In [21]:
import dagshub
import mlflow
dagshub.init(repo_owner='ldavi22', repo_name='cs229-231', mlflow=True)

mlflow.set_tracking_uri("https://dagshub.com/ldavi22/cs229-231.mlflow")
mlflow.set_experiment('house-prices-linear-regression_')

with mlflow.start_run(run_name="Decision Tree without any filtering"):

    best_idx = result_basic.best_index_
    train_rmse = np.sqrt(-result_basic.cv_results_['mean_train_score'][best_idx])
    val_rmse = np.sqrt(-result_basic.best_score_)
    val_std = result_basic.cv_results_['std_test_score'][best_idx]

    mlflow.log_param("model", "Decision Tree")
    mlflow.log_param("n_features", x_train.shape[1])
    mlflow.log_param("n_samples", x_train.shape[0])
    mlflow.log_param("feature_selection", "None")

    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("val_rmse", val_rmse)
    mlflow.log_metric("val_std", val_std)
    mlflow.log_metric("test_rmse", test_rmse_basic)
    mlflow.log_metric("overfit_gap", val_rmse - train_rmse)

    for key,value in result_basic.best_params_.items():
        mlflow.log_param(key, value)


Initialized MLflow to track repo "ldavi22/cs229-231"

Repository ldavi22/cs229-231 initialized!

🏃 View run Decision Tree without any filtering at: https://dagshub.com/ldavi22/cs229-231.mlflow/#/experiments/1/runs/1282b1978c8b42f8a0d3ea95615c1925
🧪 View experiment at: https://dagshub.com/ldavi22/cs229-231.mlflow/#/experiments/1


In [22]:
import dagshub
import mlflow
dagshub.init(repo_owner='ldavi22', repo_name='cs229-231', mlflow=True)

mlflow.set_tracking_uri("https://dagshub.com/ldavi22/cs229-231.mlflow")
mlflow.set_experiment('house-prices-linear-regression_')

with mlflow.start_run(run_name="Decision Tree with Correlation filter"):

    best_idx = result_filtered.best_index_
    train_rmse = np.sqrt(-result_filtered.cv_results_['mean_train_score'][best_idx])
    val_rmse = np.sqrt(-result_filtered.best_score_)
    val_std = result_filtered.cv_results_['std_test_score'][best_idx]

    mlflow.log_param("model", "Decision Tree")
    mlflow.log_param("n_features", x_train.shape[1])
    mlflow.log_param("n_samples", x_train.shape[0])
    mlflow.log_param("feature_selection", "Correlation Filter")

    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("val_rmse", val_rmse)
    mlflow.log_metric("val_std", val_std)
    mlflow.log_metric("test_rmse", test_rmse_filtered)

    mlflow.log_metric("overfit_gap", val_rmse - train_rmse)

    for key,value in result_filtered.best_params_.items():
        mlflow.log_param(key, value)


Initialized MLflow to track repo "ldavi22/cs229-231"

Repository ldavi22/cs229-231 initialized!

🏃 View run Decision Tree with Correlation filter at: https://dagshub.com/ldavi22/cs229-231.mlflow/#/experiments/1/runs/b7662eb0e89649db8e0d6a15423085cf
🧪 View experiment at: https://dagshub.com/ldavi22/cs229-231.mlflow/#/experiments/1
